# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoye-J/FlyRank--MachineLearning/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


Lane: Search Intent (Lane 4)
Notebook purpose: Build my lane's first five features from the March 2026
warehouse slice, then deliberately spring the leakage trap on real data: add one label-derived column, watch the score jump toward perfect, delete it, and keep the honest number.

Companion notebook: `w03_data_contract.ipynb` (ML-04).

In [1]:
# connecting to the FlyRank warehouse via Colab Secrets

from google.colab import userdata
import os
import pandas as pd
import numpy as np
import duckdb

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")

print("Connected to warehouse.")

FACT_MAR     = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
FACT_APR     = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"
FACT_MAY     = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet"
DIM_CONTENT  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

Connected to warehouse.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*


Five features,  from month=2026-03 (my mid-panel development month).
Every feature must be knowable before the decision moment.


- gsc_impressions_30d, sums only March 2026 days before any future window I predict on
- gsc_clicks_30d, same prior window
- gsc_avg_position_30d, same prior window; excludes 0 because 0 = "no data"
- sessions_ai_30d, prior-window AI-search sessions; a live signal of AI-mediated intent
- content_type, static page attribute from 'dim_content'and always knowable

Design choice:
I keep main_intent out of the 5 features on purpose. It is
the signal I want the model to help me interpret, not one of the 5 to fit.

Design rule (from the skill): a blind 'fillna(0)' injects a category signal.
For missing 'content_type', I add a 'has_content_type' flag rather than filling with a fake value. For missing GSC numerics, I keep NULL (missing ≠ zero for GSC).

In [2]:
# building the 5 feature frame from month=2026-03

FEATURES_Q = f"""
WITH perf AS (
  SELECT
    content_hash_id,
    SUM(gsc_impressions)                              AS gsc_impressions_30d,
    SUM(gsc_clicks)                                   AS gsc_clicks_30d,
    AVG(NULLIF(gsc_avg_position, 0))                  AS gsc_avg_position_30d,
    SUM(sessions_ai)                                  AS sessions_ai_30d
  FROM '{FACT_MAR}'
  GROUP BY content_hash_id
)
SELECT
  p.content_hash_id,
  p.gsc_impressions_30d,
  p.gsc_clicks_30d,
  p.gsc_avg_position_30d,
  p.sessions_ai_30d,
  d.content_type,
  d.main_intent,
  CASE WHEN d.content_type IS NULL THEN 0 ELSE 1 END AS has_content_type
FROM perf p
LEFT JOIN '{DIM_CONTENT}' d
  ON p.content_hash_id = d.content_hash_id
"""

features = con.execute(FEATURES_Q).df()
print(f"Feature frame shape: {features.shape}")
print(f"Rows with content_type present: {features['has_content_type'].mean():.1%}")
display(features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (331437, 8)
Rows with content_type present: 100.0%


,content_hash_id,gsc_impressions_30d,gsc_clicks_30d,gsc_avg_position_30d,sessions_ai_30d,content_type,main_intent,has_content_type
0,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.0,keyword article,commercial,1
1,content_05597932fe4da067,57.0,0.0,7.842593,0.0,keyword article,transactional,1
2,content_905aa32a0230694e,149.0,0.0,8.454069,0.0,keyword article,transactional,1
3,content_05434271b257bb68,1421.0,6.0,6.320337,0.0,keyword article,commercial,1
4,content_d056587ff7faca0c,2770.0,16.0,4.459107,0.0,keyword article,informational,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*



| Feature | Meaning | Missing handling | Categorical? | Available when? |
|---------|---------|------------------|--------------|------------------|
- gsc_impressions_30d, means Prior-30-day impression sum. Keep NULL, does NOT fillna(0), not categorical and available on Prior window only,always before the future window I predict on.

- gsc_clicks_30d means Prior-30-day click sum. Same missing handling, not categorical. Available Prior window.

- gsc_avg_position_30d, Prior-30-day avg position. Zeros excluded. NULL if no GSC days and not categorical.Available Prior window.

- sessions_ai_30d, Prior-30-day AI-referred sessions.0 is legitimate (no AI traffic) and not categorical, available Prior window.

- content_type, Page type from 'dim_content'. Adds has_content_type flag; do not fill. It is categorical. Static page attribute so always knowable.

Note on the 0-vs-NULL distinction:
- sessions_ai_30d = 0; the page genuinely received zero AI referrals. Keep as 0.
- gsc_impressions_30d = NULL, the page had no GSC days in March at all. That is not the same as zero impressions. Keep as NULL.

Note on the leakage risk per feature:
All five are safe **because each one only reads days inside March 2026, and my label reads April–May 2026. Nothing crosses the boundary.

In [3]:
# Inspecting missingness and cardinality of each feature

print("Missingness per feature (fraction NULL):")
print(features[[
    "gsc_impressions_30d", "gsc_clicks_30d",
    "gsc_avg_position_30d", "sessions_ai_30d",
    "content_type"
]].isna().mean().round(4).to_string())

print("\ncontent_type cardinality:")
print(features["content_type"].value_counts(dropna=False).to_string())

print("\nmain_intent cardinality (held out — not one of the 5):")
print(features["main_intent"].value_counts(dropna=False).to_string())

print("\nNumeric feature summary:")
display(features[[
    "gsc_impressions_30d", "gsc_clicks_30d",
    "gsc_avg_position_30d", "sessions_ai_30d"
]].describe().round(2))

Missingness per feature (fraction NULL):
gsc_impressions_30d     0.0000
gsc_clicks_30d          0.0000
gsc_avg_position_30d    0.4711
sessions_ai_30d         0.2133
content_type            0.0000

content_type cardinality:
content_type
keyword article       276882
feedly article         51163
comparison article      3392

main_intent cardinality (held out — not one of the 5):
main_intent
informational    185061
None              58585
transactional     46452
commercial        40158
navigational       1181

Numeric feature summary:


,gsc_impressions_30d,gsc_clicks_30d,gsc_avg_position_30d,sessions_ai_30d
count,331437.00,331437.00,175304.00,260737.00
mean,846.79,2.48,17.05,0.03
std,4044.51,19.65,18.33,0.45
min,0.00,0.00,0.10,0.00
25%,0.00,0.00,5.50,0.00
50%,2.00,0.00,9.00,0.00
75%,216.00,0.00,22.00,0.00
max,617124.00,5668.00,309.00,74.00


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*


I attack my own feature set three ways.

- Attack 1: label-derived column.
Add the raw number the label is computed from. In this warehouse, my label is
built from the *future-window* impression sum. So I add that exact future sum
back in as a column. Expect the score to jump toward 1.000.

- Attack 2: Future-window column.
Same effect, stated differently: any '*_apr_may' column measures the outcome
window. Same leak.

- Attack 3: delete the leak, keep the honest number.
Remove the leaky column. Report the honest score. That is the number I keep.

Why this matters:
From notebook 02's lesson; a model that hits 1.000 on a
leaky feature learned nothing about the world. It just learned the answer key.
The honest number is the one I carry into modeling.

In [4]:
#deliberately leak, then delete

#Step 3.1: build the future-window label from April–May 2026

LABEL_Q = f"""
WITH prior AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_mar
  FROM '{FACT_MAR}'
  GROUP BY content_hash_id
),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr_may
  FROM (
    SELECT content_hash_id, gsc_impressions FROM '{FACT_APR}'
    UNION ALL
    SELECT content_hash_id, gsc_impressions FROM '{FACT_MAY}'
  )
  GROUP BY content_hash_id
)
SELECT
  p.content_hash_id,
  p.imp_mar,
  f.imp_apr_may,
  CASE
    WHEN f.imp_apr_may IS NULL OR p.imp_mar IS NULL OR p.imp_mar = 0 THEN NULL
    WHEN f.imp_apr_may < 0.8 * p.imp_mar THEN 1
    ELSE 0
  END AS is_declining_future
FROM prior p
LEFT JOIN future f USING (content_hash_id)
"""

label_df = con.execute(LABEL_Q).df()
print(f"Label frame shape: {label_df.shape}")
print(f"Positive rate (future decline): {label_df['is_declining_future'].mean():.3f}")
print(f"NULL labels (no comparison possible): {label_df['is_declining_future'].isna().sum()}")
display(label_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Label frame shape: (331437, 4)
Positive rate (future decline): 0.283
NULL labels (no comparison possible): 154699


,content_hash_id,imp_mar,imp_apr_may,is_declining_future
0,content_4eb35f25cf28b104,0.0,4.0,<NA>
1,content_2dd03169b60f7a43,0.0,1.0,<NA>
2,content_7c9563f58a622667,0.0,1.0,<NA>
3,content_b3b255fb9aac33c1,0.0,1.0,<NA>
4,content_f431bb307481871e,0.0,3.0,<NA>


In [5]:
# Step 3.2: assemble the modeling frame

model_df = features.merge(
    label_df[["content_hash_id", "is_declining_future", "imp_apr_may"]],
    on="content_hash_id",
    how="inner",
).dropna(subset=["is_declining_future"])

print(f"Modeling frame shape: {model_df.shape}")
print(f"Positive rate: {model_df['is_declining_future'].mean():.3f}")

# One-hot content_type (kept simple for the leak demo)
X_base = model_df[[
    "gsc_impressions_30d", "gsc_clicks_30d",
    "gsc_avg_position_30d", "sessions_ai_30d",
]].fillna(0).copy()

X_base = X_base.join(
    pd.get_dummies(model_df["content_type"], prefix="ct", dummy_na=True).astype(int)
)

y = model_df["is_declining_future"].astype(int).values

print(f"\nX_base shape: {X_base.shape}")
print(f"y shape: {y.shape}, positives: {y.sum()} / {len(y)}")

Modeling frame shape: (176738, 10)
Positive rate: 0.283

X_base shape: (176738, 8)
y shape: (176738,), positives: 50017 / 176738


In [6]:
# Step 3.3: define Precision@K helper

def precision_at_k(scores, labels, k):
    """Of the top-K ranked pages by score, what fraction are truly positive?"""
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

print("Helper defined.")

Helper defined.


In [7]:
#Step 3.4: HONEST model (no leak)

from sklearn.tree import DecisionTreeClassifier, export_text

tree_honest = DecisionTreeClassifier(
    max_depth=3, class_weight="balanced", random_state=42
)
tree_honest.fit(X_base, y)

honest_scores = tree_honest.predict_proba(X_base)[:, 1]
honest_p50 = precision_at_k(honest_scores, y, 50)

print()
print("HONEST MODEL (5 clean features, no leak)")
print(f"Precision@50: {honest_p50:.3f}")
print()
print(export_text(tree_honest, feature_names=list(X_base.columns), max_depth=3))


HONEST MODEL (5 clean features, no leak)
Precision@50: 0.100

|--- gsc_clicks_30d <= 3.50
|   |--- gsc_avg_position_30d <= 12.50
|   |   |--- gsc_clicks_30d <= 0.50
|   |   |   |--- class: 1
|   |   |--- gsc_clicks_30d >  0.50
|   |   |   |--- class: 1
|   |--- gsc_avg_position_30d >  12.50
|   |   |--- gsc_impressions_30d <= 14.50
|   |   |   |--- class: 1
|   |   |--- gsc_impressions_30d >  14.50
|   |   |   |--- class: 0
|--- gsc_clicks_30d >  3.50
|   |--- gsc_clicks_30d <= 12.50
|   |   |--- gsc_avg_position_30d <= 3.15
|   |   |   |--- class: 0
|   |   |--- gsc_avg_position_30d >  3.15
|   |   |   |--- class: 0
|   |--- gsc_clicks_30d >  12.50
|   |   |--- gsc_impressions_30d <= 4992.50
|   |   |   |--- class: 0
|   |   |--- gsc_impressions_30d >  4992.50
|   |   |   |--- class: 0



In [10]:
#Step 3.5: LEAKY model(deliberately add the answer key)  + Delete the leak

X_leaky = X_base.copy()
X_leaky["imp_apr_may_LEAK"] = model_df["imp_apr_may"].fillna(0).values   # <-- THE LEAK

tree_leaky = DecisionTreeClassifier(
    max_depth=3, class_weight="balanced", random_state=42
)
tree_leaky.fit(X_leaky, y)

leaky_scores = tree_leaky.predict_proba(X_leaky)[:, 1]
leaky_p50 = precision_at_k(leaky_scores, y, 50)


print("LEAKY MODEL (5 features + imp_apr_may_LEAK)")
print()
print(f"Precision@50: {leaky_p50:.3f}  ")
print()
print(export_text(tree_leaky, feature_names=list(X_leaky.columns), max_depth=3))

print()

print(f"Honest Precision@50 : {honest_p50:.3f} (this is the number I keep)")
print(f"Leaky  Precision@50 : {leaky_p50:.3f}   (this is a mirage)")
print()
print(f"The leak inflated my score by {leaky_p50 - honest_p50:+.3f}.")
print("I delete imp_apr_may_LEAK and keep the honest number.")
print()


LEAKY MODEL (5 features + imp_apr_may_LEAK)

Precision@50: 1.000  

|--- imp_apr_may_LEAK <= 19.50
|   |--- imp_apr_may_LEAK <= 0.50
|   |   |--- class: 1
|   |--- imp_apr_may_LEAK >  0.50
|   |   |--- gsc_impressions_30d <= 1.50
|   |   |   |--- class: 0
|   |   |--- gsc_impressions_30d >  1.50
|   |   |   |--- class: 1
|--- imp_apr_may_LEAK >  19.50
|   |--- imp_apr_may_LEAK <= 1296.50
|   |   |--- gsc_impressions_30d <= 42.50
|   |   |   |--- class: 0
|   |   |--- gsc_impressions_30d >  42.50
|   |   |   |--- class: 1
|   |--- imp_apr_may_LEAK >  1296.50
|   |   |--- gsc_impressions_30d <= 1903.50
|   |   |   |--- class: 0
|   |   |--- gsc_impressions_30d >  1903.50
|   |   |   |--- class: 0


Honest Precision@50 : 0.100 (this is the number I keep)
Leaky  Precision@50 : 1.000   (this is a mirage)

The leak inflated my score by +0.900.
I delete imp_apr_may_LEAK and keep the honest number.



## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## 4. What I excluded and why

Every field I deliberately refused to use in the honest model — with one line
of why each.

Label sources (would leak)
- trend_direction, trend_pct (starter CSV only), the exact columns the
  starter label is derived from.
- is_declining_future, the label itself.
- imp_mar, imp_apr_may, the raw numbers behind the label. The _apr_may
  sum is the future window itself.
- Any *_apr_may / *_may / *_q2 column — same reason.

Pseudonyms (grouping only)
- content_hash_id, client_hash_id, join and split keys; never features.

Sealed data
- All rows from month=2026-06, the natural outcome window of any
  past to future label. Reserved as a sealed test month.
- fact_content_daily_performance_sample.parquet, the sample is exactly
  June 2026, not random.

Flags where zeros are fill
- ga4_* where ga4_data_available = FALSE; zeros are fill, not measurement.
- gsc_* where gsc_data_available = FALSE, same.

Product-decision metadata
- is_published, is_deleted (dim_content): decision flags.
- provider_used, model_used (dim_content): internal product metadata.
- last_optimized_date, optimization_eligible_date: workflow fields that
  encode prior product decisions.

Repeated-per-row context (from fact_content_query_90d)
- Its per-content context columns must be aggregated with ANY_VALUE(), never SUM(), that would double-count.

In [12]:
# explicit list of what was excluded and why

excluded = [
    ("trend_direction",         "starter CSV only, label source"),
    ("trend_pct",               "starter CSV only, label source"),
    ("is_declining_future",     "the label itself"),
    ("imp_mar",                 "the prior-window number behind the label"),
    ("imp_apr_may",             "the future-window number behind the label"),
    ("*_may, *_q2, *_apr_may",  "future window"),
    ("content_hash_id",         "pseudonym, join/split only"),
    ("client_hash_id",          "pseudonym, split only"),
    ("month=2026-06",           "sealed test month"),
    ("_sample.parquet",         "final month only, not random"),
    ("ga4_* when unavailable",  "zeros are fill, not measurement"),
    ("gsc_* when unavailable",  "zeros are fill, not measurement"),
    ("is_published, is_deleted","product-decision flag"),
    ("provider_used, model_used","internal product metadata"),
    ("last_optimized_date",     "workflow field, prior product decision"),
    ("optimization_eligible_date","workflow field, prior product decision"),
    ("query_90d context cols",  "use ANY_VALUE(), never SUM()"),
]

print(f"{'FIELD':<28} WHY EXCLUDED")
print("-" * 80)
for col, why in excluded:
    print(f"{col:<28} {why}")

FIELD                        WHY EXCLUDED
--------------------------------------------------------------------------------
trend_direction              starter CSV only, label source
trend_pct                    starter CSV only, label source
is_declining_future          the label itself
imp_mar                      the prior-window number behind the label
imp_apr_may                  the future-window number behind the label
*_may, *_q2, *_apr_may       future window
content_hash_id              pseudonym, join/split only
client_hash_id               pseudonym, split only
month=2026-06                sealed test month
_sample.parquet              final month only, not random
ga4_* when unavailable       zeros are fill, not measurement
gsc_* when unavailable       zeros are fill, not measurement
is_published, is_deleted     product-decision flag
provider_used, model_used    internal product metadata
last_optimized_date          workflow field, prior product decision
optimization_eligib

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.